# 01 · 随机变量与概率分布 Random Variables & Probability Distributions

> **能力标签**：SH6（概率与统计 / Probability & statistics）

## 目标 Objectives

完成本课后，你应该能够：

1. 理解**随机变量**（random variable）的概念及其期望（expectation）与方差（variance）的定义。
2. 掌握四种常见分布：均匀（Uniform）、正态（Normal）、伯努利（Bernoulli）、二项（Binomial）。
3. 用 `np.random.default_rng` 采样并绘制直方图（Agg 后端，保存到临时目录）。

> 对应能力：**SH6**


## 概念讲解 Concepts

### 随机变量 Random Variable

**随机变量** $X$ 是将样本空间映射到实数的函数。其统计特性由两个核心量描述：

| 量 | 离散 | 连续 |
|---|---|---|
| **期望** $E[X]$ | $\sum_x x \cdot P(X=x)$ | $\int x f(x)\,dx$ |
| **方差** $\text{Var}(X)$ | $E[(X-\mu)^2]$ | $E[(X-\mu)^2]$ |

方差 $\text{Var}(X) = E[X^2] - (E[X])^2$。

---

### 四种常见分布

#### 均匀分布 Uniform Distribution
$X \sim \text{Uniform}(a,b)$：在区间 $[a,b]$ 上等概率取值。
- $E[X] = (a+b)/2$，$\text{Var}(X) = (b-a)^2/12$

#### 正态分布 Normal Distribution
$X \sim \mathcal{N}(\mu, \sigma^2)$：钟形曲线，描述自然测量误差。
- $E[X] = \mu$，$\text{Var}(X) = \sigma^2$

#### 伯努利分布 Bernoulli Distribution
$X \sim \text{Bernoulli}(p)$：单次试验结果为 0 或 1（成功概率 $p$）。
- $E[X] = p$，$\text{Var}(X) = p(1-p)$

#### 二项分布 Binomial Distribution
$X \sim \text{Binomial}(n, p)$：$n$ 次独立伯努利试验的成功次数。
- $E[X] = np$，$\text{Var}(X) = np(1-p)$
- $P(X=k) = \binom{n}{k} p^k (1-p)^{n-k}$

---

### 采样与直方图 Sampling & Histograms

用 `np.random.default_rng(seed)` 采样，确保可复现：

```python
rng = np.random.default_rng(42)
samples = rng.normal(loc=0, scale=1, size=1000)
```

直方图将样本密度可视化，当样本量足够大时，形状逼近理论概率密度函数（PDF）。


## 示例 Worked Example

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import tempfile
from pathlib import Path
from scipy import stats

rng = np.random.default_rng(42)
n = 2000

# 采样四种分布
s_uniform   = rng.uniform(0, 1, n)
s_normal    = rng.normal(0, 1, n)
s_bernoulli = rng.binomial(1, 0.3, n).astype(float)   # Bernoulli(p=0.3)
s_binomial  = rng.binomial(20, 0.4, n).astype(float)  # Binomial(n=20, p=0.4)

samples = [s_uniform, s_normal, s_bernoulli, s_binomial]
titles  = ["Uniform(0,1)", "Normal(0,1)", "Bernoulli(p=0.3)", "Binomial(n=20,p=0.4)"]

fig, axes = plt.subplots(1, 4, figsize=(14, 3))
fig.suptitle("四种常见分布的采样直方图", fontsize=13)

for ax, s, title in zip(axes, samples, titles):
    ax.hist(s, bins=30, density=True, color="steelblue", edgecolor="white", alpha=0.8)
    ax.set_title(title, fontsize=9)
    ax.set_xlabel("值")
    ax.set_ylabel("密度")

# 叠加正态 PDF
x = np.linspace(-4, 4, 200)
axes[1].plot(x, stats.norm.pdf(x, 0, 1), "r-", lw=2, label="理论PDF")
axes[1].legend(fontsize=8)

fig.tight_layout()
_tmpdir = tempfile.mkdtemp()
_outpath = Path(_tmpdir) / "distributions_demo.png"
fig.savefig(_outpath, dpi=80)
plt.close(fig)
print("图像已保存到:", _outpath)

# 计算期望与方差（理论 vs 样本）
print("\n=== 期望 (E[X]) vs 样本均值 ===")
expected = [0.5, 0.0, 0.3, 8.0]
for name, s, e in zip(titles, samples, expected):
    print(f"  {name}: 理论={e:.2f}, 样本={s.mean():.4f}")

print("\n=== 方差 (Var(X)) vs 样本方差 ===")
variances = [1/12, 1.0, 0.3*0.7, 20*0.4*0.6]
for name, s, v in zip(titles, samples, variances):
    print(f"  {name}: 理论={v:.4f}, 样本={s.var():.4f}")


## 动手 Exercise

In [ ]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import tempfile
from pathlib import Path

# 练习：生成泊松分布样本并可视化
# 泊松分布 Poisson(lambda=5)：E[X] = lambda, Var(X) = lambda
rng = np.random.default_rng(99)
n = 1000
lam = 5

samples_poisson = rng.poisson(lam, n)

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(samples_poisson, bins=range(0, 20), density=True,
        color="mediumpurple", edgecolor="white", alpha=0.8,
        label=f"Poisson(λ={lam}) 样本")
ax.axvline(samples_poisson.mean(), color="red", ls="--", label=f"样本均值={samples_poisson.mean():.2f}")
ax.set_title(f"Poisson(λ={lam}) 分布 (n={n})")
ax.set_xlabel("k（事件次数）")
ax.set_ylabel("概率密度")
ax.legend()
fig.tight_layout()

_tmpdir = tempfile.mkdtemp()
_outpath = Path(_tmpdir) / "poisson_exercise.png"
fig.savefig(_outpath, dpi=80)
plt.close(fig)
print("图像已保存到:", _outpath)

print(f"\n泊松(λ={lam}) 统计量:")
print(f"  理论期望 = {lam:.2f}, 样本均值 = {samples_poisson.mean():.4f}")
print(f"  理论方差 = {lam:.2f}, 样本方差 = {samples_poisson.var():.4f}")


## 延伸阅读 Further Reading

1. **概率论入门（Khan Academy）**：<https://www.khanacademy.org/math/statistics-probability>
2. **NumPy random Generator 文档**：<https://numpy.org/doc/stable/reference/random/generator.html>
3. **《All of Statistics》 — Larry Wasserman**：第 1–3 章（离散与连续分布）
4. **scipy.stats 分布列表**：<https://docs.scipy.org/doc/scipy/reference/stats.html>


## 关联练习 Related Assignments

本课无直接自动评分包，作为后续练习的基础概念铺垫。后续课程关联包：

- **`w4-descriptive`**：描述统计（均值/方差/中位数）
- **`w4-clt`**：抽样分布与中心极限定理

```bash
python check.py w4-descriptive
```

> 能力标签：**SH6** · threshold ≥ 0.7
